In [1]:
import onnx

model_path = r'C:\Users\swest\Desktop\traffic-capturing\traffic-reporting\models\weights\fixed_cupola_6120_640px_bs64_dynamic_ultralytics.onnx'

try:
    # Load the ONNX model
    model = onnx.load(model_path)

    input_names = [inp.name for inp in model.graph.input]

    print(f"Definitive Model Input Name(s) found in '{model_path}':")
    if not input_names:
        print("No primary inputs found. This might indicate an issue with the model or it might only have initializer inputs.")
    else:
        for name in input_names:
            print(f"- {name}")

    # For more details, you can print shape information if available
    print("\nDetailed Input Information:")
    for i, input_tensor in enumerate(model.graph.input):
        print(f"\nInput #{i}:")
        print(f"  Name: {input_tensor.name}")
        # The shape is under type.tensor_type.shape.dim
        dims = input_tensor.type.tensor_type.shape.dim
        shape_str = ", ".join([str(dim.dim_value) if dim.dim_value > 0 else dim.dim_param if dim.dim_param else "?" for dim in dims])
        print(f"  Shape (from model): [{shape_str}]")
        print(f"  Data Type (from model): {onnx.TensorProto.DataType.Name(input_tensor.type.tensor_type.elem_type)}")

except Exception as e:
    print(f"An error occurred: {e}")
    print("Please ensure the model path is correct and the ONNX file is not corrupted.")

Definitive Model Input Name(s) found in 'C:\Users\swest\Desktop\traffic-capturing\traffic-reporting\models\weights\fixed_cupola_6120_640px_bs64_dynamic_ultralytics.onnx':
- images

Detailed Input Information:

Input #0:
  Name: images
  Shape (from model): [batch, 3, height, width]
  Data Type (from model): FLOAT


In [2]:
import onnx

model = onnx.load(model_path)

# Check input dimensions
for input in model.graph.input:
    print(f"Input: {input.name}")
    for dim in input.type.tensor_type.shape.dim:
        if dim.dim_param:
            print(f"  Dynamic dimension: {dim.dim_param}")
        else:
            print(f"  Fixed dimension: {dim.dim_value}")


Input: images
  Dynamic dimension: batch
  Fixed dimension: 3
  Dynamic dimension: height
  Dynamic dimension: width


In [3]:
import onnxruntime as ort
import numpy as np

session = ort.InferenceSession(model_path, providers=['CUDAExecutionProvider'])

# Test batch size 1 with correct dtype
input_1 = np.random.randn(1, 3, 640, 640).astype(np.float16)  # Changed to float16
output_1 = session.run(None, {'images': input_1})

# Test batch size 4
input_4 = np.random.randn(4, 3, 640, 640).astype(np.float16)  # Changed to float16
output_4 = session.run(None, {'images': input_4})
print("Dynamic batch size confirmed!")
print(f"Batch 1 output shape: {output_1[0].shape}")
print(f"Batch 4 output shape: {output_4[0].shape}")


InvalidArgument: [ONNXRuntimeError] : 2 : INVALID_ARGUMENT : Unexpected input data type. Actual: (tensor(float16)) , expected: (tensor(float))

In [ ]:
import ast
props = { p.key : p.value for p in model.metadata_props }
if 'names' in props:
    names = ast.literal_eval(props['names'])
names

{0: 'Light Vehicle',
 1: 'Bus',
 2: 'Pedestrian',
 3: 'Short Truck',
 4: 'Medium Truck',
 5: 'Long Truck',
 6: 'Cyclist',
 7: 'Motorcycle',
 8: 'Minibus Taxi'}

In [5]:
import os
base_dir = r'C:\Users\swest\Desktop\traffic-capturing\traffic-reporting\models\weights'
for model_name in os.listdir(base_dir):
    if not model_name.endswith('.onnx'): continue
    weights_path = f"{base_dir}/{model_name}"
    print('Model:',model_name)
    m = onnx.load(weights_path)
    props = { p.key : p.value for p in model.metadata_props }
    print('Class Names:',props['names'])
    print('*'*100)
    print('-'*100)


Model: fixed_cupola_6120_640px.onnx
Class Names: {0: 'Light Vehicle', 1: 'Bus', 2: 'Pedestrian', 3: 'Short Truck', 4: 'Medium Truck', 5: 'Long Truck', 6: 'Cyclist', 7: 'Motorcycle', 8: 'Minibus Taxi'}
****************************************************************************************************
----------------------------------------------------------------------------------------------------
Model: fixed_cupola_6120_640px_bs64_dynamic_ultralytics.onnx
Class Names: {0: 'Light Vehicle', 1: 'Bus', 2: 'Pedestrian', 3: 'Short Truck', 4: 'Medium Truck', 5: 'Long Truck', 6: 'Cyclist', 7: 'Motorcycle', 8: 'Minibus Taxi'}
****************************************************************************************************
----------------------------------------------------------------------------------------------------
Model: SGDMed_SGD_LR0p0100_WD0p00050_Cls1p50_Mix0p15_20250418_112036.onnx
Class Names: {0: 'Light Vehicle', 1: 'Bus', 2: 'Pedestrian', 3: 'Short Truck', 4: 'Medium Truck